In [1]:
#full step 2 data
import json
with open('/scratch/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.28,22:10-5.json', 'r') as f:
    full_data = json.load(f)
len(full_data)

23325

In [2]:
#upload 7b step 2 data
import json
from datasets import Dataset

with open('/scratch/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.28,22:10-5.json', 'r') as f:
    step2_full_raw = json.load(f)

step2_full_raw_filtered = []

for data in step2_full_raw:
    try:
        # Get the first available index (0-4) for each field
        question = next((data[f'Question_{i}'] for i in range(5) if f'Question_{i}' in data), None)
        answer = next((data[f'Answer_{i}'] for i in range(5) if f'Answer_{i}' in data), None)
        output = next((data[f'Output_{i}'] for i in range(5) if f'Output_{i}' in data), None)
        output_tokens = next((data[f'output_tokens_{i}'] for i in range(5) if f'output_tokens_{i}' in data), None)
        pred_answer = next((data[f'Pred_Answer_{i}'] for i in range(5) if f'Pred_Answer_{i}' in data), None)
        metrics = next((data[f'Metrics_{i}'] for i in range(5) if f'Metrics_{i}' in data), None)
        
        if all(v is not None for v in [question, answer, output, output_tokens, pred_answer, metrics]):
            step2_full_raw_filtered.append({
                'id': data['new_id'],
                'Question': question,
                'Answer': answer,
                'Output': output,
                'Output_tokens': output_tokens,
                'Pred_Answer': pred_answer,
                'Metrics': metrics
            })
    except Exception as e:
        print(f"Error processing item {data.get('new_id', 'unknown')}: {str(e)}")
        continue

print(f"Processed {len(step2_full_raw_filtered)} items successfully")

dataset = Dataset.from_list(step2_full_raw_filtered)
dataset.push_to_hub('talzoomanzoo/0609-7B-step2-full-raw', private=True)

/home/hojinkim/miniconda3/envs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processed 23325 items successfully


Creating parquet from Arrow format: 100%|██████████| 12/12 [00:00<00:00, 82.67ba/s]
/home/hojinkim/miniconda3/envs/py310/lib/python3.10/site-packages/huggingface_hub/lfs.py:337: UserWarning: hf_transfer is enabled but does not support uploading from bytes or BinaryIO, falling back to regular upload
  warnings.warn(
Uploading the dataset shards: 100%|██████████| 2/2 [00:12<00:00,  6.13s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0609-7B-step2-full-raw/commit/ff028b09f2ac8e661a5f56d7b5fe48e6faa067f5', commit_message='Upload dataset', commit_description='', oid='ff028b09f2ac8e661a5f56d7b5fe48e6faa067f5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0609-7B-step2-full-raw', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0609-7B-step2-full-raw'), pr_revision=None, pr_num=None)

In [4]:
#validation set using test set
import json
with open("/scratch/hojinkim/search-o1-dev/data/AIME/test.json", "r") as f:
    validation_set = json.load(f)

In [5]:
#1. 0603_dpo_z_ytrue, 0603_dpo_zy_ytrue, 0603_e2e_zy_ytrue
import json
import random

with open('/home/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.28,22:10-5.json', 'r') as f:
    rft_data = json.load(f)

rft_z_data = []

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

# First, separate data into math_equal=True and math_equal=False groups
true_examples = []
false_examples = []

for data in rft_data:
    has_math_equal = any(
        key.startswith('Metrics_') and data[key].get('math_equal', False)
        for key in data
    )
    
    entry = {
        'idx': len(true_examples if has_math_equal else false_examples) + 1,
        'id': data.get('new_id', None),
        'input_x': '',
        'ground_truth': '',
        'output_z': '',
        'output_y': '',
        'metrics': []
    }

    # Collect all matching fields
    for key in data:
        if key.startswith('Question_'):
            entry['input_x'] = extract_x(data[key])
            entry['output_z'] = "<think>" + extract_z(data[key])
        elif key.startswith('Answer_'):
            entry['ground_truth'] = data[key]
        elif key.startswith('Output_'):
            entry['output_y'] = "</think>" + data[key]
        elif key.startswith('Metrics_'):
            entry['metrics'].append(data[key])
    
    if has_math_equal:
        true_examples.append(entry)
    else:
        false_examples.append(entry)

# Create pairs of examples
count = 0
for true_ex in true_examples:
    for false_ex in false_examples:
        if true_ex['input_x'] == false_ex['input_x']:  # Only pair examples with same input
            count += 1
            rft_entry = {
                'idx': count,
                'id': f"{true_ex['id']}_{false_ex['id']}",
                'input_x': true_ex['input_x'],
                'ground_truth': true_ex['ground_truth'],
                'chosen_z': true_ex['output_z'],
                'rejected_z': false_ex['output_z'],
                'chosen_y': true_ex['output_y'],
                'rejected_y': false_ex['output_y'],
                'chosen_zy': true_ex['output_z'] + true_ex['output_y'],
                'rejected_zy': false_ex['output_z'] + false_ex['output_y'],
                'metrics': true_ex['metrics'] + false_ex['metrics']
            }
            rft_z_data.append(rft_entry)

random.seed(42)
random.shuffle(rft_z_data)
rft_z_data = rft_z_data[:1000]

with open('0603.rft_z_data.json', 'w') as f:
    json.dump(rft_z_data, f, ensure_ascii=False, indent=4)
print(f"Created {len(rft_z_data)} RFT pairs")
print(f"Total true examples: {len(true_examples)}")
print(f"Total false examples: {len(false_examples)}")



Created 1000 RFT pairs
Total true examples: 9486
Total false examples: 13839


In [7]:
#huggingface push 0603_dpo_z_ytrue, 0603_dpo_zy_ytrue, 0603_e2e_zy_ytrue
from datasets import Dataset

dataset = Dataset.from_list(rft_z_data)
dataset.push_to_hub('talzoomanzoo/0603-dpo-z-ytrue', private=True, split="train")

dataset.push_to_hub('talzoomanzoo/0603-dpo-zy-ytrue', private=True, split="train")

dataset.push_to_hub('talzoomanzoo/0603-e2e-zy-ytrue', private=True, split="train")

Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.04it/s]
No files have been modified since last commit. Skipping to prevent empty commit.
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.31it/s]


ValueError: Features of the new split don't match the features of the existing splits on the hub: {'Question': Value(dtype='string', id=None), 'answer': Value(dtype='string', id=None)} != {'idx': Value(dtype='int64', id=None), 'id': Value(dtype='string', id=None), 'input_x': Value(dtype='string', id=None), 'ground_truth': Value(dtype='string', id=None), 'chosen_z': Value(dtype='string', id=None), 'rejected_z': Value(dtype='string', id=None), 'chosen_y': Value(dtype='string', id=None), 'rejected_y': Value(dtype='string', id=None), 'chosen_zy': Value(dtype='string', id=None), 'rejected_zy': Value(dtype='string', id=None), 'metrics': [{'is_valid_answer': Value(dtype='bool', id=None), 'math_equal': Value(dtype='bool', id=None)}]}

In [6]:
#2. 0603_rft_z_ytrue, 0603_rft_zy_ytrue
import json
import random

with open('/home/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.28,22:10-5.json', 'r') as f:
    rft_data = json.load(f)

rft_z_data = []

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

# First, separate data into math_equal=True and math_equal=False groups
true_examples = []
false_examples = []

for data in rft_data:
    has_math_equal = any(
        key.startswith('Metrics_') and data[key].get('math_equal', False)
        for key in data
    )
    
    entry = {
        'idx': len(true_examples if has_math_equal else false_examples) + 1,
        'id': data.get('new_id', None),
        'input_x': '',
        'ground_truth': '',
        'output_z': '',
        'output_y': '',
        'metrics': []
    }

    # Collect all matching fields
    for key in data:
        if key.startswith('Question_'):
            entry['input_x'] = extract_x(data[key])
            entry['output_z'] = "<think>" + extract_z(data[key])
        elif key.startswith('Answer_'):
            entry['ground_truth'] = data[key]
        elif key.startswith('Output_'):
            entry['output_y'] = "</think>" + data[key]
        elif key.startswith('Metrics_'):
            entry['metrics'].append(data[key])
    
    if has_math_equal:
        true_examples.append(entry)
    else:
        false_examples.append(entry)

# Create pairs of examples
count = 0
for true_ex in true_examples:
    count += 1
    rft_entry = {
                'idx': count,
                'id': f"{true_ex['id']}",
                'input_x': true_ex['input_x'],
                'ground_truth': true_ex['ground_truth'],
                'chosen_z': true_ex['output_z'],
                'chosen_y': true_ex['output_y'],
                'chosen_zy': true_ex['output_z'] + true_ex['output_y'],
                'metrics': true_ex['metrics']
            }
    rft_z_data.append(rft_entry)

random.seed(42)
random.shuffle(rft_z_data)
rft_z_data = rft_z_data[:1000]

with open('0603.rft_z_data.json', 'w') as f:
    json.dump(rft_z_data, f, ensure_ascii=False, indent=4)
print(f"Created {len(rft_z_data)} RFT pairs")
print(f"Total true examples: {len(true_examples)}")
print(f"Total false examples: {len(false_examples)}")



Created 1000 RFT pairs
Total true examples: 9486
Total false examples: 13839


In [7]:
#huggingface push 0603_rft_z_ytrue, 0603_rft_zy_ytrue
from datasets import Dataset

dataset = Dataset.from_list(rft_z_data)
dataset.push_to_hub('talzoomanzoo/0603-rft-z-ytrue', private=True)

dataset.push_to_hub('talzoomanzoo/0603-rft-zy-ytrue', private=True)


Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.38s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0603-rft-zy-ytrue/commit/c0f2d989c91723237c23a88586a41ac48af46fe0', commit_message='Upload dataset', commit_description='', oid='c0f2d989c91723237c23a88586a41ac48af46fe0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0603-rft-zy-ytrue', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0603-rft-zy-ytrue'), pr_revision=None, pr_num=None)

In [7]:
#3. dpo-z-zthreshold
import json
import re

with open('/home/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.28,22:10-5.json', 'r') as f:
    z_y_combined_data = json.load(f)

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

filtered_z_y_combined_data = []

# Process data in chunks of 25 rows
for idx in range(0, len(z_y_combined_data), 25):
    chunk_25 = z_y_combined_data[idx:idx+25]
    subgroup_data = []  # Store (percentage, data) pairs for each subgroup
    subgroup_z_chosen_y_chosen = []
    subgroup_z_chosen_y_rejected = []
    subgroup_z_rejected_y_chosen = []
    subgroup_z_rejected_y_rejected = []
    
    # Process each subgroup of 5 rows within the 25-row chunk
    for i in range(0, len(chunk_25), 5):
        subgroup = chunk_25[i:i+5]
        true_pairs = []
        false_pairs = []
        
        # Count true and false pairs in this subgroup
        for data in subgroup:
            for key in data:
                if key.startswith('Metrics_'):
                    if data[key].get('math_equal', True):
                        true_pairs.append(data)
                    else:
                        false_pairs.append(data)
        
        # Calculate z_true_percentage for this subgroup
        z_true_percentage = len(true_pairs) / len(subgroup)
        
        # Store the percentage and corresponding data
        if 0.4 <= z_true_percentage < 1.0 and true_pairs:
            subgroup_data.append((z_true_percentage, true_pairs[0], false_pairs[0], 'chosen'))
        elif 0 < z_true_percentage < 0.4 and false_pairs:
            subgroup_data.append((z_true_percentage, true_pairs[0], false_pairs[0], 'rejected'))
    
    # If we have data, find the highest and lowest percentage entries
    if subgroup_data:
        # Sort by percentage
        subgroup_data.sort(key=lambda x: x[0])
        
        # Get the highest percentage (chosen) and lowest percentage (rejected)
        chosen_z_data = None
        rejected_z_data = None
        chosen_z_chosen_y_data = None
        chosen_z_rejected_y_data = None
        rejected_z_chosen_y_data = None
        rejected_z_rejected_y_data = None
        
        for percentage, true_data, false_data, type_ in subgroup_data:
            if type_ == 'chosen' and chosen_z_data is None:
                chosen_z_data = true_data
                chosen_z_chosen_y_data = true_data
                chosen_z_rejected_y_data = false_data
            elif type_ == 'rejected' and rejected_z_data is None:
                rejected_z_data = false_data
                rejected_z_chosen_y_data = true_data
                rejected_z_rejected_y_data = false_data
        
        # If we have both chosen and rejected data, create an entry
        if chosen_z_data and rejected_z_data:
            #x, z_chosen, y_chosen under z _chosen, y_reject under z _chosen, z_reject, y_chosen under z _reject, y_reject under z _reject 
            entry = {
                'idx': idx+1,
                'id': chosen_z_data.get('new_id', None).split('_')[0],
                'input_x': '',
                'chosen_z': '',
                'chosen_y_under_chosen_z': '',
                'rejected_y_under_chosen_z': '',
                'rejected_z': '',
                'chosen_y_under_rejected_z': '',
                'rejected_y_under_rejected_z': ''
            }
            
            # Process chosen z data
            for key in chosen_z_data:
                if key.startswith('Question_'):
                    entry['input_x'] = extract_x(chosen_z_data[key])
                    entry['chosen_z'] = "<think>" + extract_z(chosen_z_data[key])
                if key.startswith('Answer_'):
                    entry['ground_truth'] = chosen_z_data[key]
            for key in chosen_z_chosen_y_data:
                if key.startswith('Output_'):
                    entry['chosen_y_under_chosen_z'] = "</think>" + chosen_z_chosen_y_data[key]
            for key in chosen_z_rejected_y_data:
                if key.startswith('Output_'):
                    entry['rejected_y_under_chosen_z'] = "</think>" + chosen_z_rejected_y_data[key]
            
            # Process rejected z data
            for key in rejected_z_data:
                if key.startswith('Question_'):
                    entry['rejected_z'] = "<think>" + extract_z(rejected_z_data[key])
            for key in rejected_z_chosen_y_data:
                if key.startswith('Output_'):
                    entry['chosen_y_under_rejected_z'] = "</think>" + rejected_z_chosen_y_data[key]
            for key in rejected_z_rejected_y_data:
                if key.startswith('Output_'):
                    entry['rejected_y_under_rejected_z'] = "</think>" + rejected_z_rejected_y_data[key]
            
            filtered_z_y_combined_data.append(entry)

with open('0603-zy-combined-data.json', 'w') as f:
    json.dump(filtered_z_y_combined_data, f, ensure_ascii=False, indent=4)

print(len(filtered_z_y_combined_data))

714


In [1]:
#4. 0603_dpo_y_ytrue
import json
import random

with open('/scratch/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.28,22:10-5.json', 'r') as f:
    dpo_data = json.load(f)

dpo_y_data = []

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

# First, separate data into math_equal=True and math_equal=False groups
true_examples = []
false_examples = []

for data in dpo_data:
    has_math_equal = any(
        key.startswith('Metrics_') and data[key].get('math_equal', False)
        for key in data
    )
    
    entry = {
        'idx': len(true_examples if has_math_equal else false_examples) + 1,
        'id': data.get('new_id', None),
        'input_x': '',
        'ground_truth': '',
        'output_z': '',
        'output_y': '',
        'metrics': []
    }

    # Collect all matching fields
    for key in data:
        if key.startswith('Question_'):
            entry['input_x'] = extract_x(data[key])
            entry['output_z'] = "<think>" + extract_z(data[key])
        elif key.startswith('Answer_'):
            entry['ground_truth'] = data[key]
        elif key.startswith('Output_'):
            entry['output_y'] = "</think>" + data[key]
        elif key.startswith('Metrics_'):
            entry['metrics'].append(data[key])
    
    if has_math_equal:
        true_examples.append(entry)
    else:
        false_examples.append(entry)

# Create pairs of examples
count = 0
for true_ex in true_examples:
    for false_ex in false_examples:
        if true_ex['input_x'] + true_ex['output_z'] == false_ex['input_x'] + false_ex['output_z']:  # Only pair examples with same input
            count += 1
            dpo_entry = {
                'idx': count,
                'id': f"{true_ex['id']}_{false_ex['id']}",
                'input_x_z': true_ex['input_x'] + true_ex['output_z'],
                'ground_truth': true_ex['ground_truth'],
                'output_z': true_ex['output_z'],
                'chosen_y': true_ex['output_y'],
                'rejected_y': false_ex['output_y'],
                'metrics': true_ex['metrics'] + false_ex['metrics']
            }
            dpo_y_data.append(dpo_entry)

random.seed(42)
random.shuffle(dpo_y_data)
dpo_y_data = dpo_y_data[:1000]

with open('0603.dpo_y_data.json', 'w') as f:
    json.dump(dpo_y_data, f, ensure_ascii=False, indent=4)
print(f"Created {len(dpo_y_data)} DPO pairs")
print(f"Total true examples: {len(true_examples)}")
print(f"Total false examples: {len(false_examples)}")

Created 1000 DPO pairs
Total true examples: 9486
Total false examples: 13839


In [2]:
#huggingface push dpo-y-ytrue
from datasets import Dataset

dataset = Dataset.from_list(dpo_y_data)
dataset.push_to_hub('talzoomanzoo/0603-dpo-y-ytrue', private=True)

/home/hojinkim/miniconda3/envs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.20it/s]
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0603-dpo-y-ytrue/commit/ac808335e95d1a2c3495f4c5d4601b9e28ee48a8', commit_message='Upload dataset', commit_description='', oid='ac808335e95d1a2c3495f4c5d4601b9e28ee48a8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0603-dpo-y-ytrue', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0603-dpo-y-ytrue'), pr_revision=None, pr_num=None)

In [1]:
#4. 0603_dpo_zy_ytrue
import json
import random

with open('/scratch/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.28,22:10-5.json', 'r') as f:
    dpo_data = json.load(f)

dpo_zy_data = []

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

# First, separate data into math_equal=True and math_equal=False groups
true_examples = []
false_examples = []

for data in dpo_data:
    has_math_equal = any(
        key.startswith('Metrics_') and data[key].get('math_equal', False)
        for key in data
    )
    
    entry = {
        'idx': len(true_examples if has_math_equal else false_examples) + 1,
        'id': data.get('new_id', None),
        'input_x': '',
        'ground_truth': '',
        'output_z': '',
        'output_y': '',
        'metrics': []
    }

    # Collect all matching fields
    for key in data:
        if key.startswith('Question_'):
            entry['input_x'] = extract_x(data[key])
            entry['output_z'] = "<think>" + extract_z(data[key])
        elif key.startswith('Answer_'):
            entry['ground_truth'] = data[key]
        elif key.startswith('Output_'):
            entry['output_y'] = "</think>" + data[key]
        elif key.startswith('Metrics_'):
            entry['metrics'].append(data[key])
    
    if has_math_equal:
        true_examples.append(entry)
    else:
        false_examples.append(entry)

# Create pairs of examples
count = 0
for true_ex in true_examples:
    for false_ex in false_examples:
        if true_ex['input_x'] == false_ex['input_x']:  # Only pair examples with same input
            count += 1
            dpo_entry = {
                'idx': count,
                'id': f"{true_ex['id']}_{false_ex['id']}",
                'input_x': true_ex['input_x'],
                'ground_truth': true_ex['ground_truth'],
                'chosen_z_y': true_ex['output_z'] + true_ex['output_y'],
                'rejected_z_y': false_ex['output_z'] + false_ex['output_y'],
                'metrics': true_ex['metrics'] + false_ex['metrics']
            }
            dpo_zy_data.append(dpo_entry)

random.seed(42)
random.shuffle(dpo_zy_data)
dpo_zy_data = dpo_zy_data[:1000]

with open('0603.dpo_zy_data.json', 'w') as f:
    json.dump(dpo_zy_data, f, ensure_ascii=False, indent=4)
print(f"Created {len(dpo_zy_data)} DPO pairs")
print(f"Total true examples: {len(true_examples)}")
print(f"Total false examples: {len(false_examples)}")

Created 1000 DPO pairs
Total true examples: 9486
Total false examples: 13839


In [2]:
#huggingface push dpo-zy-ytrue
from datasets import Dataset

dataset = Dataset.from_list(dpo_zy_data)
dataset.push_to_hub('talzoomanzoo/0603-dpo-zy-ytrue', private=True)
dataset.push_to_hub('talzoomanzoo/0603-rft-zy-ytrue', private=True)

/home/hojinkim/miniconda3/envs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.21it/s]
No files have been modified since last commit. Skipping to prevent empty commit.
Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.56s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0603-rft-zy-ytrue/commit/220ec9e477d023a496adeb179de60154c1c2b64c', commit_message='Upload dataset', commit_description='', oid='220ec9e477d023a496adeb179de60154c1c2b64c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0603-rft-zy-ytrue', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0603-rft-zy-ytrue'), pr_revision=None, pr_num=None)

In [3]:
#3. dpo-zy-zthreshold
import json
import re

with open('/scratch/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.28,22:10-5.json', 'r') as f:
    z_y_combined_data = json.load(f)

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

filtered_z_y_combined_data = []

# Process data in chunks of 25 rows
for idx in range(0, len(z_y_combined_data), 25):
    chunk_25 = z_y_combined_data[idx:idx+25]
    subgroup_data = []  # Store (percentage, data) pairs for each subgroup
    subgroup_z_chosen_y_chosen = []
    subgroup_z_chosen_y_rejected = []
    subgroup_z_rejected_y_chosen = []
    subgroup_z_rejected_y_rejected = []
    
    # Process each subgroup of 5 rows within the 25-row chunk
    for i in range(0, len(chunk_25), 5):
        subgroup = chunk_25[i:i+5]
        true_pairs = []
        false_pairs = []
        
        # Count true and false pairs in this subgroup
        for data in subgroup:
            for key in data:
                if key.startswith('Metrics_'):
                    if data[key].get('math_equal', True):
                        true_pairs.append(data)
                    else:
                        false_pairs.append(data)
        
        # Calculate z_true_percentage for this subgroup
        z_true_percentage = len(true_pairs) / len(subgroup)
        
        # Store the percentage and corresponding data
        if 0.4 <= z_true_percentage < 1.0 and true_pairs:
            subgroup_data.append((z_true_percentage, true_pairs[0], false_pairs[0], 'chosen'))
        elif 0 < z_true_percentage < 0.4 and false_pairs:
            subgroup_data.append((z_true_percentage, true_pairs[0], false_pairs[0], 'rejected'))
    
    # If we have data, find the highest and lowest percentage entries
    if subgroup_data:
        # Sort by percentage
        subgroup_data.sort(key=lambda x: x[0])
        
        # Get the highest percentage (chosen) and lowest percentage (rejected)
        chosen_z_data = None
        rejected_z_data = None
        chosen_z_chosen_y_data = None
        chosen_z_rejected_y_data = None
        rejected_z_chosen_y_data = None
        rejected_z_rejected_y_data = None
        
        for percentage, true_data, false_data, type_ in subgroup_data:
            if type_ == 'chosen' and chosen_z_data is None:
                chosen_z_data = true_data
                chosen_z_chosen_y_data = true_data
                chosen_z_rejected_y_data = false_data
            elif type_ == 'rejected' and rejected_z_data is None:
                rejected_z_data = false_data
                rejected_z_chosen_y_data = true_data
                rejected_z_rejected_y_data = false_data
        
        # If we have both chosen and rejected data, create an entry
        if chosen_z_data and rejected_z_data:
            #x, z_chosen, y_chosen under z _chosen, y_reject under z _chosen, z_reject, y_chosen under z _reject, y_reject under z _reject 
            entry = {
                'idx': idx+1,
                'id': chosen_z_data.get('new_id', None).split('_')[0],
                'input_x': '',
                'chosen_z': '',
                'chosen_y_under_chosen_z': '',
                'rejected_y_under_chosen_z': '',
                'rejected_z': '',
                'chosen_y_under_rejected_z': '',
                'rejected_y_under_rejected_z': '',
                'chosen_z_y': '',
                'rejected_z_y': ''
            }
            
            # Process chosen z data
            for key in chosen_z_data:
                if key.startswith('Question_'):
                    entry['input_x'] = extract_x(chosen_z_data[key])
                    entry['chosen_z'] = "<think>" + extract_z(chosen_z_data[key])
                if key.startswith('Answer_'):
                    entry['ground_truth'] = chosen_z_data[key]
            for key in chosen_z_chosen_y_data:
                if key.startswith('Output_'):
                    entry['chosen_y_under_chosen_z'] = "</think>" + chosen_z_chosen_y_data[key]
                    entry['chosen_z_y'] = entry['chosen_z'] + entry['chosen_y_under_chosen_z']
            for key in chosen_z_rejected_y_data:
                if key.startswith('Output_'):
                    entry['rejected_y_under_chosen_z'] = "</think>" + chosen_z_rejected_y_data[key]
            
            # Process rejected z data
            for key in rejected_z_data:
                if key.startswith('Question_'):
                    entry['rejected_z'] = "<think>" + extract_z(rejected_z_data[key])
            for key in rejected_z_chosen_y_data:
                if key.startswith('Output_'):
                    entry['chosen_y_under_rejected_z'] = "</think>" + rejected_z_chosen_y_data[key]
            for key in rejected_z_rejected_y_data:
                if key.startswith('Output_'):
                    entry['rejected_y_under_rejected_z'] = "</think>" + rejected_z_rejected_y_data[key]
                    entry['rejected_z_y'] = entry['rejected_z'] + entry['rejected_y_under_rejected_z']
            filtered_z_y_combined_data.append(entry)

with open('0603-zy-combined-data.json', 'w') as f:
    json.dump(filtered_z_y_combined_data, f, ensure_ascii=False, indent=4)

print(len(filtered_z_y_combined_data))

714


In [4]:
#huggingface push dpo-zy-zthreshold
from datasets import Dataset

dataset = Dataset.from_list(filtered_z_y_combined_data)
dataset.push_to_hub('talzoomanzoo/0603-dpo-zy-zthreshold', private=True)

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  4.12ba/s]
/home/hojinkim/miniconda3/envs/py310/lib/python3.10/site-packages/huggingface_hub/lfs.py:337: UserWarning: hf_transfer is enabled but does not support uploading from bytes or BinaryIO, falling back to regular upload
  warnings.warn(
Uploading the dataset shards: 100%|██████████| 1/1 [00:05<00:00,  5.66s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0603-dpo-zy-zthreshold/commit/1cb63a729c1a41128ea7dd392c109c3cd32becd3', commit_message='Upload dataset', commit_description='', oid='1cb63a729c1a41128ea7dd392c109c3cd32becd3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0603-dpo-zy-zthreshold', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0603-dpo-zy-zthreshold'), pr_revision=None, pr_num=None)

In [1]:
#4. rft-zy-zthreshold
import json
import re

with open('/scratch/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.28,22:10-5.json', 'r') as f:
    z_y_combined_data = json.load(f)

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

filtered_z_y_combined_data = []

# Process data in chunks of 25 rows
for idx in range(0, len(z_y_combined_data), 25):
    chunk_25 = z_y_combined_data[idx:idx+25]
    subgroup_data = []  # Store (percentage, data) pairs for each subgroup
    subgroup_z_chosen_y_chosen = []
    subgroup_z_chosen_y_rejected = []
    subgroup_z_rejected_y_chosen = []
    subgroup_z_rejected_y_rejected = []
    
    # Process each subgroup of 5 rows within the 25-row chunk
    for i in range(0, len(chunk_25), 5):
        subgroup = chunk_25[i:i+5]
        true_pairs = []
        false_pairs = []
        
        # Count true and false pairs in this subgroup
        for data in subgroup:
            for key in data:
                if key.startswith('Metrics_'):
                    if data[key].get('math_equal', True):
                        true_pairs.append(data)
                    else:
                        false_pairs.append(data)
        
        # Calculate z_true_percentage for this subgroup
        z_true_percentage = len(true_pairs) / len(subgroup)
        
        # Store the percentage and corresponding data
        if 0.4 <= z_true_percentage < 1.0 and true_pairs:
            subgroup_data.append((z_true_percentage, true_pairs[0], false_pairs[0], 'chosen'))
        elif 0 < z_true_percentage < 0.4 and false_pairs:
            subgroup_data.append((z_true_percentage, true_pairs[0], false_pairs[0], 'rejected'))
    
    # If we have data, find the highest and lowest percentage entries
    if subgroup_data:
        # Sort by percentage
        subgroup_data.sort(key=lambda x: x[0])
        
        # Get the highest percentage (chosen) and lowest percentage (rejected)
        chosen_z_data = None
        rejected_z_data = None
        chosen_z_chosen_y_data = None
        chosen_z_rejected_y_data = None
        rejected_z_chosen_y_data = None
        rejected_z_rejected_y_data = None
        
        for percentage, true_data, false_data, type_ in subgroup_data:
            if type_ == 'chosen' and chosen_z_data is None:
                chosen_z_data = true_data
                chosen_z_chosen_y_data = true_data
                chosen_z_rejected_y_data = false_data
            elif type_ == 'rejected' and rejected_z_data is None:
                rejected_z_data = false_data
                rejected_z_chosen_y_data = true_data
                rejected_z_rejected_y_data = false_data
        
        # If we have both chosen and rejected data, create an entry
        if chosen_z_data and rejected_z_data:
            #x, z_chosen, y_chosen under z _chosen, y_reject under z _chosen, z_reject, y_chosen under z _reject, y_reject under z _reject 
            entry = {
                'idx': idx+1,
                'id': chosen_z_data.get('new_id', None).split('_')[0],
                'input_x': '',
                'chosen_z': '',
                'chosen_zy': '',
                'chosen_y_under_chosen_z': '',
                'rejected_y_under_chosen_z': '',
                'rejected_z': '',
                'rejected_zy': '',
                'chosen_y_under_rejected_z': '',
                'rejected_y_under_rejected_z': ''
            }
            
            # Process chosen z data
            for key in chosen_z_data:
                if key.startswith('Question_'):
                    entry['input_x'] = extract_x(chosen_z_data[key])
                    entry['chosen_z'] = "<think>" + extract_z(chosen_z_data[key])
                if key.startswith('Answer_'):
                    entry['ground_truth'] = chosen_z_data[key]
            for key in chosen_z_chosen_y_data:
                if key.startswith('Output_'):
                    entry['chosen_y_under_chosen_z'] = "</think>" + chosen_z_chosen_y_data[key]
                    entry['chosen_zy'] = entry['chosen_z'] + entry['chosen_y_under_chosen_z']
            for key in chosen_z_rejected_y_data:
                if key.startswith('Output_'):
                    entry['rejected_y_under_chosen_z'] = "</think>" + chosen_z_rejected_y_data[key]
            
            # Process rejected z data
            for key in rejected_z_data:
                if key.startswith('Question_'):
                    entry['rejected_z'] = "<think>" + extract_z(rejected_z_data[key])
            for key in rejected_z_chosen_y_data:
                if key.startswith('Output_'):
                    entry['chosen_y_under_rejected_z'] = "</think>" + rejected_z_chosen_y_data[key]
            for key in rejected_z_rejected_y_data:
                if key.startswith('Output_'):
                    entry['rejected_y_under_rejected_z'] = "</think>" + rejected_z_rejected_y_data[key]
                    entry['rejected_zy'] = entry['rejected_z'] + entry['rejected_y_under_rejected_z']
            filtered_z_y_combined_data.append(entry)

with open('0603-zy-combined-data.json', 'w') as f:
    json.dump(filtered_z_y_combined_data, f, ensure_ascii=False, indent=4)

print(len(filtered_z_y_combined_data))

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7feb614b3370>>
Traceback (most recent call last):
  File "/home/hojinkim/miniconda3/envs/py310/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


714


In [3]:
#huggingface push rft-zy-zthreshold
from datasets import Dataset

dataset = Dataset.from_list(filtered_z_y_combined_data)
dataset.push_to_hub('talzoomanzoo/0603-rft-zy-zthreshold', private=True)

/home/hojinkim/miniconda3/envs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  7.57ba/s]
/home/hojinkim/miniconda3/envs/py310/lib/python3.10/site-packages/huggingface_hub/lfs.py:337: UserWarning: hf_transfer is enabled but does not support uploading from bytes or BinaryIO, falling back to regular upload
  warnings.warn(
Uploading the dataset shards: 100%|██████████| 1/1 [00:05<00:00,  5.56s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0603-rft-zy-zthreshold/commit/a0ef3a23e8909df6357492ef5b6e0cb12a43971a', commit_message='Upload dataset', commit_description='', oid='a0ef3a23e8909df6357492ef5b6e0cb12a43971a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0603-rft-zy-zthreshold', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0603-rft-zy-zthreshold'), pr_revision=None, pr_num=None)

In [4]:
#5. 0603_rft_y_ytrue
import json
import random

with open('/scratch/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.28,22:10-5.json', 'r') as f:
    rft_data = json.load(f)

rft_y_data = []

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

# First, separate data into math_equal=True and math_equal=False groups
true_examples = []
false_examples = []

for data in rft_data:
    has_math_equal = any(
        key.startswith('Metrics_') and data[key].get('math_equal', False)
        for key in data
    )
    
    entry = {
        'idx': len(true_examples if has_math_equal else false_examples) + 1,
        'id': data.get('new_id', None),
        'input_x': '',
        'ground_truth': '',
        'output_z': '',
        'output_y': '',
        'metrics': []
    }

    # Collect all matching fields
    for key in data:
        if key.startswith('Question_'):
            entry['input_x'] = extract_x(data[key])
            entry['output_z'] = "<think>" + extract_z(data[key])
        elif key.startswith('Answer_'):
            entry['ground_truth'] = data[key]
        elif key.startswith('Output_'):
            entry['output_y'] = "</think>" + data[key]
        elif key.startswith('Metrics_'):
            entry['metrics'].append(data[key])
    
    if has_math_equal:
        true_examples.append(entry)
    else:
        false_examples.append(entry)

# Create pairs of examples
count = 0
for true_ex in true_examples:
    for false_ex in false_examples:
        if true_ex['input_x'] + true_ex['output_z'] == false_ex['input_x'] + false_ex['output_z']:  # Only pair examples with same input
            count += 1
            rft_y_entry = {
                'idx': count,
                'id': f"{true_ex['id']}_{false_ex['id']}",
                'input_x_z': true_ex['input_x'] + true_ex['output_z'],
                'ground_truth': true_ex['ground_truth'],
                'output_z': true_ex['output_z'],
                'chosen_y': true_ex['output_y'],
                'rejected_y': false_ex['output_y'],
                'metrics': true_ex['metrics'] + false_ex['metrics']
            }
            rft_y_data.append(rft_y_entry)

random.seed(42)
random.shuffle(rft_y_data)
rft_y_data = rft_y_data[:1000]

with open('0603.rft_y_data.json', 'w') as f:
    json.dump(rft_y_data, f, ensure_ascii=False, indent=4)
print(f"Created {len(rft_y_data)} RFT pairs")
print(f"Total true examples: {len(true_examples)}")
print(f"Total false examples: {len(false_examples)}")

Created 1000 RFT pairs
Total true examples: 9486
Total false examples: 13839


In [5]:
#huggingface push rft-y-ytrue
from datasets import Dataset

dataset = Dataset.from_list(rft_y_data)
dataset.push_to_hub('talzoomanzoo/0603-rft-y-ytrue', private=True)

Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.16s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0603-rft-y-ytrue/commit/e1e1aa5d10cf5e2bb0f3d99d8086b80f9ce4c9a9', commit_message='Upload dataset', commit_description='', oid='e1e1aa5d10cf5e2bb0f3d99d8086b80f9ce4c9a9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0603-rft-y-ytrue', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0603-rft-y-ytrue'), pr_revision=None, pr_num=None)